In [6]:
# %%
# ============================================================
# 연령×성별 교차 층화 기반 당뇨·고혈압 예측 및
# K-means 복합건강등급 산정 파이프라인
#
# 데이터  : 국민건강영양조사(KNHANES) 2020–2024
# 층화    : 연령군(중장년 40–59세 / 고령 60세+) × 성별(남/여) → 4개 모델
# 알고리즘: RF, LightGBM, XGBoost, MLP, TabNet (성능 비교 후 최적 선택)
# XAI     : SHAP (전역 변수 중요도 + 개별 관측치 해석)
# 등급    : K-means 기반 데이터 주도 복합건강등급
# ============================================================

# %%
# ## 0. 라이브러리 불러오기

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat
import joblib
import shap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score
)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import xgboost as xgb
import optuna
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 한글 폰트 설정 (논문 그래프용)
# Windows: 'Malgun Gothic' / Mac: 'AppleGothic' / Linux: 'NanumGothic'
plt.rcParams['font.family'] = 'Malgun Gothic'
# 마이너스 부호 깨짐 방지: matplotlib 기본 폰트에서 '-' 렌더링
plt.rcParams['axes.unicode_minus'] = False
# SHAP 내부 텍스트(axis tick 등)도 동일 폰트 적용
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# 재현성 고정
SEED = 42
np.random.seed(SEED)

# 4개 층화 그룹 정의
AGEGROUP_CONFIG = {
    "중장년_남성": {"age_min": 40, "age_max": 59, "sex_code": 1.0, "age_group": 0.0},
    "중장년_여성": {"age_min": 40, "age_max": 59, "sex_code": 2.0, "age_group": 0.0},
    "고령_남성":   {"age_min": 60, "age_max": 99, "sex_code": 1.0, "age_group": 1.0},
    "고령_여성":   {"age_min": 60, "age_max": 99, "sex_code": 2.0, "age_group": 1.0},
}

# %%
# ## 1. KNHANES 2020–2024 데이터 병합

df20, _ = pyreadstat.read_sas7bdat('hn20_all.sas7bdat')
df21, _ = pyreadstat.read_sas7bdat('hn21_all.sas7bdat')
df22, _ = pyreadstat.read_sas7bdat('hn22_all.sas7bdat')
df23, _ = pyreadstat.read_sas7bdat('hn23_all.sas7bdat')
df24, _ = pyreadstat.read_sas7bdat('hn24_all.sas7bdat')

# %%
# ### 1-1. 분석 변수 선택
# 공모전 논문(연구1) 변수 체계 기반 + 연령(age) 추가
# 건강검진 없이 설문만으로 답변 가능한 항목으로 구성

KEY_COLS = ['ID', 'year', 'sex', 'age']

CAT_COLS = [
    'HE_obe',    # 비만도
    'BO1_1',     # 체중변화 상태
    'BO1_2',     # 체중감소량
    'BO1_3',     # 체중증가량
    'BD1_11',    # 음주빈도 (1년간)
    'BD2_1',     # 1회 음주량
    'BS3_1',     # 흡연상태
    'BE3_71',    # 격렬한 신체활동 (직업)
    'BE3_75',    # 격렬한 신체활동 (여가)
    'BE3_81',    # 중등도 신체활동 (직업)
    'BE3_91',    # 걷기 활동
    'pa_aerobic',# 유산소 신체활동 실천율
    'L_BR_FQ',   # 아침식사 빈도
    'BP1',       # 스트레스 인지
    'mh_stress', # 스트레스 인지율
    'incm',      # 개인소득 분위
    'ho_incm',   # 가구소득 분위
    'edu',       # 교육수준
    'BH1',       # 건강검진 수검 여부
]

NUM_COLS = [
    'HE_BMI',   # 체질량지수
    'HE_wc',    # 허리둘레
    'HE_wt',    # 체중
    'N_EN',     # 에너지 섭취량 (kcal)
    'N_CHO',    # 탄수화물 (g)
    'N_SUGAR',  # 당류 (g)
    'N_NA',     # 나트륨 (mg)
    'N_FAT',    # 지방 (g)
    'N_SFA',    # 포화지방산 (g)
    'N_TDF',    # 식이섬유 (g)
    'N_K',      # 칼륨 (mg)
    'N_PROT',   # 단백질 (g)
]

# 종속변수: 당뇨(HbA1c 기준) + 고혈압
TARGET_COLS = ['HE_DM_HbA1c', 'HE_HP']

ALL_VARS = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

# %%
# ### 1-2. 연도별 데이터 병합 (존재하는 컬럼만 선택)

df_total = pd.concat(
    [d[[v for v in ALL_VARS if v in d.columns]].copy()
     for d in [df20, df21, df22, df23, df24]],
    axis=0
).reset_index(drop=True)

print(f"병합 후 전체 행수: {len(df_total):,}")
print(f"컬럼 수: {df_total.shape[1]}")

# %%
# ### 1-3. 종속변수 이진 인코딩
# HE_DM_HbA1c: 1=정상 → 0, 3=당뇨 → 1
# HE_HP:        1=정상 → 0, 4=고혈압 → 1

df_total['HE_DM_HbA1c'] = df_total['HE_DM_HbA1c'].map({1: 0, 3: 1})
df_total['HE_HP']        = df_total['HE_HP'].map({1: 0, 4: 1})

# 결측 제거 (종속변수 기준)
df_total = df_total.dropna(subset=['HE_DM_HbA1c', 'HE_HP']).reset_index(drop=True)

print("\n당뇨 분포:\n", df_total['HE_DM_HbA1c'].value_counts())
print("\n고혈압 분포:\n", df_total['HE_HP'].value_counts())

# %%
# ### 1-4. 연령 전처리 및 층화 변수 생성
# 분석 대상: 40세 이상 (보험 언더라이팅 실무 기준)
# 중장년(40–59세) = 0 / 고령(60세+) = 1

df_total['age'] = pd.to_numeric(df_total['age'], errors='coerce')
df_total = df_total[df_total['age'] >= 40].copy().reset_index(drop=True)
df_total['age_group'] = np.where(df_total['age'] < 60, 0, 1)

print("\n연령군 분포 (0=중장년 40–59세, 1=고령 60세+):")
print(df_total['age_group'].value_counts().sort_index())

# %%
# ### 1-5. 컬럼명 영문 레이블로 변환

LABEL_DICT = {
    'ID': 'ID', 'year': 'SurveyYear', 'sex': 'Sex',
    'age': 'Age', 'age_group': 'AgeGroup',
    'HE_DM_HbA1c': 'Diabetes', 'HE_HP': 'Hypertension',
    'HE_obe':    'ObesityStatus',
    'BO1_1':     'WeightChangeStatus',
    'BO1_2':     'WeightLossAmount',
    'BO1_3':     'WeightGainAmount',
    'BD1_11':    'DrinkingFrequency',
    'BD2_1':     'DrinkingAmount',
    'BS3_1':     'SmokingStatus',
    'BE3_71':    'VigorousActivity_Work',
    'BE3_75':    'VigorousActivity_Leisure',
    'BE3_81':    'ModerateActivity_Work',
    'BE3_91':    'WalkingActivity',
    'pa_aerobic':'AerobicActivityRate',
    'L_BR_FQ':   'BreakfastFrequency',
    'BP1':       'StressLevel',
    'mh_stress': 'StressAwarenessRate',
    'incm':      'PersonalIncomeQuartile',
    'ho_incm':   'HouseholdIncomeQuartile',
    'edu':       'EducationLevel',
    'BH1':       'HealthScreeningStatus',
    'HE_BMI':    'BMI', 'HE_wc': 'WaistCirc', 'HE_wt': 'Weight',
    'N_EN':      'Energy_kcal', 'N_CHO': 'Carb_g',
    'N_SUGAR':   'Sugar_g',     'N_NA':  'Sodium_mg',
    'N_FAT':     'Fat_g',       'N_SFA': 'SaturatedFat_g',
    'N_TDF':     'Fiber_g',     'N_K':   'Potassium_mg',
    'N_PROT':    'Protein_g',
}

df_total.rename(columns=LABEL_DICT, inplace=True)
df_total.to_csv('knhanes_preprocessed.csv', index=False, encoding='utf-8-sig')
print(">>> 전처리 완료. 저장: knhanes_preprocessed.csv")

# %%
# ## 2. 피처 정의

CAT_FEATURES = [
    'ObesityStatus', 'WeightChangeStatus', 'WeightLossAmount', 'WeightGainAmount',
    'DrinkingFrequency', 'DrinkingAmount', 'SmokingStatus',
    'VigorousActivity_Work', 'VigorousActivity_Leisure',
    'ModerateActivity_Work', 'WalkingActivity', 'AerobicActivityRate',
    'BreakfastFrequency', 'StressLevel', 'StressAwarenessRate',
    'PersonalIncomeQuartile', 'HouseholdIncomeQuartile',
    'EducationLevel', 'HealthScreeningStatus',
]

NUM_FEATURES = [
    'BMI', 'WaistCirc', 'Weight',
    'Energy_kcal', 'Carb_g', 'Sugar_g', 'Sodium_mg',
    'Fat_g', 'SaturatedFat_g', 'Fiber_g', 'Potassium_mg', 'Protein_g',
]

X_FEATURES = NUM_FEATURES + CAT_FEATURES

# 수치형 데이터셋 구성 (결측 0 대체)
required_cols = X_FEATURES + ['Diabetes', 'Hypertension', 'Sex', 'AgeGroup']
df_final = df_total[[c for c in required_cols if c in df_total.columns]].copy()
for col in df_final.columns:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
df_final = df_final.astype(float)

print(f"\ndf_final 형태: {df_final.shape}")

# 연령군 × 성별 × 질환 분포 확인 (논문 표 1 기초 자료)
print("\n[연령군 × 성별 × 당뇨 분포]")
print(df_final.groupby(['AgeGroup', 'Sex'])['Diabetes'].value_counts().unstack(fill_value=0))
print("\n[연령군 × 성별 × 고혈압 분포]")
print(df_final.groupby(['AgeGroup', 'Sex'])['Hypertension'].value_counts().unstack(fill_value=0))

# %%
# ## 3. 알고리즘별 성능 비교 함수
# RF, LightGBM, XGBoost, MLP, TabNet 5개 알고리즘
# 평가지표: AUC, F1(가중), Precision, Recall
# Optuna 하이퍼파라미터 최적화 (AUC 기준)

def train_and_evaluate(X_tr, X_val, y_tr, y_val, algo: str, n_trials: int = 20):
    """
    단일 알고리즘 학습 및 검증 성능 반환.
    algo: 'RF' | 'LGBM' | 'XGB' | 'MLP' | 'TabNet'
    """
    sw = compute_sample_weight('balanced', y=y_tr)

    # ── Random Forest ─────────────────────────────────────────
    if algo == 'RF':
        def obj(trial):
            m = RandomForestClassifier(
                n_estimators = trial.suggest_int('n_estimators', 100, 500),
                max_depth    = trial.suggest_int('max_depth', 3, 15),
                min_samples_split = trial.suggest_int('min_samples_split', 2, 10),
                random_state = SEED, n_jobs=-1,
            )
            m.fit(X_tr, y_tr, sample_weight=sw)
            return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=n_trials)
        model = RandomForestClassifier(
            **study.best_params, random_state=SEED, n_jobs=-1
        )
        model.fit(X_tr, y_tr, sample_weight=sw)

    # ── LightGBM ──────────────────────────────────────────────
    elif algo == 'LGBM':
        def obj(trial):
            m = lgb.LGBMClassifier(
                n_estimators  = trial.suggest_int('n_estimators', 100, 500),
                max_depth     = trial.suggest_int('max_depth', 3, 7),
                learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                subsample     = trial.suggest_float('subsample', 0.7, 1.0),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0),
                random_state  = SEED, n_jobs=-1, verbose=-1,
            )
            m.fit(X_tr, y_tr, sample_weight=sw)
            return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=n_trials)
        model = lgb.LGBMClassifier(
            **study.best_params, random_state=SEED, n_jobs=-1, verbose=-1
        )
        model.fit(X_tr, y_tr, sample_weight=sw)

    # ── XGBoost ───────────────────────────────────────────────
    elif algo == 'XGB':
        def obj(trial):
            m = xgb.XGBClassifier(
                n_estimators  = trial.suggest_int('n_estimators', 100, 500),
                max_depth     = trial.suggest_int('max_depth', 3, 7),
                learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                subsample     = trial.suggest_float('subsample', 0.7, 1.0),
                colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0),
                use_label_encoder=False, eval_metric='logloss',
                random_state=SEED, tree_method='hist',
            )
            m.fit(X_tr, y_tr, sample_weight=sw)
            return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=n_trials)
        model = xgb.XGBClassifier(
            **study.best_params,
            use_label_encoder=False, eval_metric='logloss',
            random_state=SEED, tree_method='hist',
        )
        model.fit(X_tr, y_tr, sample_weight=sw)

    # ── MLP ───────────────────────────────────────────────────
    elif algo == 'MLP':
        def obj(trial):
            n_layers = trial.suggest_int('n_layers', 1, 3)
            n_units  = trial.suggest_int('n_units', 32, 256)
            hidden   = tuple([n_units] * n_layers)
            m = MLPClassifier(
                hidden_layer_sizes  = hidden,
                alpha               = trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                learning_rate_init  = trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
                max_iter=300, random_state=SEED,
            )
            m.fit(X_tr, y_tr)
            return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=n_trials)
        bp = study.best_params
        model = MLPClassifier(
            hidden_layer_sizes = tuple([bp['n_units']] * bp['n_layers']),
            alpha              = bp['alpha'],
            learning_rate_init = bp['learning_rate_init'],
            max_iter=300, random_state=SEED,
        )
        model.fit(X_tr, y_tr)

    # ── TabNet ────────────────────────────────────────────────
    # optimizer_params 딕셔너리로 학습률 전달 (pytorch-tabnet API 규격)
    elif algo == 'TabNet':
        def obj(trial):
            lr = trial.suggest_float('lr', 1e-3, 1e-1, log=True)
            m  = TabNetClassifier(
                n_d     = trial.suggest_int('n_d', 8, 64),
                n_a     = trial.suggest_int('n_a', 8, 64),
                n_steps = trial.suggest_int('n_steps', 3, 10),
                gamma   = trial.suggest_float('gamma', 1.0, 2.0),
                optimizer_params = {'lr': lr},
                seed=SEED, verbose=0,
                device_name='cuda' if torch.cuda.is_available() else 'cpu',
            )
            m.fit(
                X_train=X_tr.values, y_train=y_tr.values,
                eval_set=[(X_val.values, y_val.values)],
                max_epochs=100, patience=10, batch_size=1024,
                weights=1,
            )
            return roc_auc_score(y_val, m.predict_proba(X_val.values)[:, 1])
        study = optuna.create_study(direction='maximize')
        study.optimize(obj, n_trials=n_trials)
        bp = study.best_params
        model = TabNetClassifier(
            n_d=bp['n_d'], n_a=bp['n_a'], n_steps=bp['n_steps'],
            gamma=bp['gamma'],
            optimizer_params={'lr': bp['lr']},
            seed=SEED, verbose=0,
            device_name='cuda' if torch.cuda.is_available() else 'cpu',
        )
        model.fit(
            X_train=X_tr.values, y_train=y_tr.values,
            max_epochs=200, patience=20, batch_size=1024, weights=1,
        )
    else:
        raise ValueError(f"지원하지 않는 알고리즘: {algo}")

    # ── 성능 지표 계산 ────────────────────────────────────────
    if algo == 'TabNet':
        y_prob = model.predict_proba(X_val.values)[:, 1]
        y_pred = model.predict(X_val.values)
    else:
        y_prob = model.predict_proba(X_val)[:, 1]
        y_pred = model.predict(X_val)

    auc       = roc_auc_score(y_val, y_prob)
    f1        = f1_score(y_val, y_pred, average='weighted')
    from sklearn.metrics import precision_score, recall_score
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)

    return model, {
        'AUC': round(auc, 4),
        'F1':  round(f1,  4),
        'Precision': round(precision, 4),
        'Recall':    round(recall,    4),
    }

# %%
# ## 4. 4개 층화 그룹 × 2개 질환 × 5개 알고리즘 학습 (논문 표 2)

ALGOS = ['RF', 'LGBM', 'XGB', 'MLP', 'TabNet']
DISEASES = {
    'Diabetes':     '당뇨',
    'Hypertension': '고혈압',
}

# 결과 저장용
all_results    = {}   # {그룹명: {질환: {알고리즘: metrics}}}
best_models    = {}   # {그룹명: {질환: 최적모델}}
best_algo_name = {}   # {그룹명: {질환: 최적알고리즘명}}

for grp_name, cfg in AGEGROUP_CONFIG.items():
    all_results[grp_name] = {}
    best_models[grp_name] = {}
    best_algo_name[grp_name] = {}

    # 해당 그룹 서브셋 추출
    df_g = df_final[
        (df_final['AgeGroup'] == cfg['age_group']) &
        (df_final['Sex']      == cfg['sex_code'])
    ].copy()
    print(f"\n{'='*20} [{grp_name}] 표본 수: {len(df_g):,}명 {'='*20}")

    for dis_col, dis_name in DISEASES.items():
        print(f"\n  ── [{dis_name}] 알고리즘 비교 ──")
        all_results[grp_name][dis_col] = {}

        X = df_g[X_FEATURES]
        y = df_g[dis_col].astype(int)

        # 7:3 분할 (층화)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=0.3, random_state=SEED, stratify=y
        )

        best_auc   = -1
        best_model = None
        best_algo  = None

        for algo in ALGOS:
            try:
                model, metrics = train_and_evaluate(X_tr, X_val, y_tr, y_val, algo)
                all_results[grp_name][dis_col][algo] = metrics
                print(f"    {algo:8s} | AUC={metrics['AUC']:.4f} "
                      f"F1={metrics['F1']:.4f} "
                      f"Prec={metrics['Precision']:.4f} "
                      f"Rec={metrics['Recall']:.4f}")
                if metrics['AUC'] > best_auc:
                    best_auc   = metrics['AUC']
                    best_model = model
                    best_algo  = algo
            except Exception as e:
                print(f"    {algo:8s} | 오류: {e}")
                all_results[grp_name][dis_col][algo] = None

        best_models[grp_name][dis_col]    = best_model
        best_algo_name[grp_name][dis_col] = best_algo
        print(f"  → 최적 알고리즘: {best_algo} (AUC={best_auc:.4f})")

# 성능 결과 CSV 저장 (hold-out 결과)
rows = []
for grp, dis_dict in all_results.items():
    for dis, algo_dict in dis_dict.items():
        for algo, metrics in algo_dict.items():
            if metrics:
                rows.append({'그룹': grp, '질환': dis, '알고리즘': algo, **metrics})
pd.DataFrame(rows).to_csv('성능비교_holdout.csv', index=False, encoding='utf-8-sig')
print("\n>>> Hold-out 성능 비교 결과 저장: 성능비교_holdout.csv")

# %%
# ## 4-1. 5-fold 교차검증 (논문 표 2 완성용)
# hold-out AUC 최적 알고리즘에 대해서만 CV 실행 (속도·안정성 균형)
# 평가지표: AUC (macro OvR), F1-weighted, Precision-weighted, Recall-weighted
# TabNet은 sklearn CV 인터페이스 미지원 → 수동 fold 루프로 처리

from sklearn.metrics import precision_score, recall_score

def run_cv(df_grp: pd.DataFrame, algo: str, disease: str,
           n_splits: int = 5) -> dict:
    """
    5-fold 층화 교차검증.
    반환: {지표명: 'mean ± std'} 딕셔너리
    """
    X = df_grp[X_FEATURES]
    y = df_grp[disease].astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    fold_auc, fold_f1, fold_prec, fold_rec = [], [], [], []

    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        sw = compute_sample_weight('balanced', y=y_tr)

        # ── 알고리즘별 모델 생성 (최적 파라미터 재탐색 없이 고정 파라미터 사용)
        # 이미 hold-out에서 선정된 best_params 재활용
        bp = params_dict.get(grp_name, {}).get(disease, {})

        if algo == 'RF':
            mdl = RandomForestClassifier(
                **{k: v for k, v in bp.items()
                   if k in ['n_estimators', 'max_depth', 'min_samples_split']},
                random_state=SEED, n_jobs=-1,
            )
            mdl.fit(X_tr, y_tr, sample_weight=sw)
            y_prob = mdl.predict_proba(X_val)[:, 1]

        elif algo == 'LGBM':
            mdl = lgb.LGBMClassifier(
                **{k: v for k, v in bp.items()
                   if k in ['n_estimators', 'max_depth', 'learning_rate',
                             'subsample', 'colsample_bytree']},
                random_state=SEED, n_jobs=-1, verbose=-1,
            )
            mdl.fit(X_tr, y_tr, sample_weight=sw)
            y_prob = mdl.predict_proba(X_val)[:, 1]

        elif algo == 'XGB':
            mdl = xgb.XGBClassifier(
                **{k: v for k, v in bp.items()
                   if k in ['n_estimators', 'max_depth', 'learning_rate',
                             'subsample', 'colsample_bytree']},
                use_label_encoder=False, eval_metric='logloss',
                random_state=SEED, tree_method='hist',
            )
            mdl.fit(X_tr, y_tr, sample_weight=sw)
            y_prob = mdl.predict_proba(X_val)[:, 1]

        elif algo == 'MLP':
            mdl = MLPClassifier(
                hidden_layer_sizes=tuple(
                    [bp.get('n_units', 128)] * bp.get('n_layers', 2)
                ),
                alpha=bp.get('alpha', 1e-4),
                learning_rate_init=bp.get('learning_rate_init', 1e-3),
                max_iter=300, random_state=SEED,
            )
            mdl.fit(X_tr, y_tr)
            y_prob = mdl.predict_proba(X_val)[:, 1]

        elif algo == 'TabNet':
            # TabNet: sklearn CV 미지원 → 수동 fold 처리
            mdl = TabNetClassifier(
                n_d=bp.get('n_d', 32), n_a=bp.get('n_a', 32),
                n_steps=bp.get('n_steps', 5),
                gamma=bp.get('gamma', 1.5),
                optimizer_params={'lr': bp.get('lr', 2e-2)},
                seed=SEED, verbose=0,
                device_name='cuda' if torch.cuda.is_available() else 'cpu',
            )
            mdl.fit(
                X_train=X_tr.values, y_train=y_tr.values,
                max_epochs=200, patience=20,
                batch_size=1024, weights=1,
            )
            y_prob = mdl.predict_proba(X_val.values)[:, 1]
        else:
            continue

        y_pred = (y_prob >= 0.5).astype(int)
        fold_auc.append(roc_auc_score(y_val, y_prob))
        fold_f1.append(f1_score(y_val, y_pred, average='weighted', zero_division=0))
        fold_prec.append(precision_score(y_val, y_pred, average='weighted', zero_division=0))
        fold_rec.append(recall_score(y_val, y_pred, average='weighted', zero_division=0))

    def fmt(arr):
        return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"

    return {
        'CV_AUC':       fmt(fold_auc),
        'CV_F1':        fmt(fold_f1),
        'CV_Precision': fmt(fold_prec),
        'CV_Recall':    fmt(fold_rec),
        # 원시값도 보관 (유의성 검정 등 후속 분석용)
        '_raw_auc':  fold_auc,
    }

# best_params 저장 딕셔너리 (CV fold 내 재사용)
params_dict = {}   # {그룹명: {질환: best_params}}

# hold-out 학습 시 best_params를 함께 저장하도록 all_results 재활용
# → train_and_evaluate 반환값에 params 추가 필요 없이,
#   아래에서 각 그룹×질환의 최적 알고리즘 파라미터를 Optuna에서 다시 가져옴
# (이미 best_models에 학습된 모델이 있으므로 파라미터 재탐색 없이 모델 내부에서 추출)

for grp_name in AGEGROUP_CONFIG:
    params_dict[grp_name] = {}
    for dis_col in DISEASES:
        mdl = best_models.get(grp_name, {}).get(dis_col)
        if mdl is None:
            params_dict[grp_name][dis_col] = {}
            continue
        # sklearn/lgbm/xgb 모델은 get_params()로 추출
        if hasattr(mdl, 'get_params'):
            params_dict[grp_name][dis_col] = mdl.get_params()
        else:
            # TabNet은 직접 속성 참조
            params_dict[grp_name][dis_col] = {
                'n_d': mdl.n_d, 'n_a': mdl.n_a,
                'n_steps': mdl.n_steps, 'gamma': mdl.gamma,
                'lr': mdl.optimizer_params.get('lr', 2e-2),
            }

# 5-fold CV 실행 (최적 알고리즘만)
print("\n" + "="*60)
print("5-fold 교차검증 (최적 알고리즘 기준)")
print("="*60)

cv_rows = []
for grp_name, cfg in AGEGROUP_CONFIG.items():
    df_g = df_final[
        (df_final['AgeGroup'] == cfg['age_group']) &
        (df_final['Sex']      == cfg['sex_code'])
    ].copy()

    for dis_col, dis_name in DISEASES.items():
        algo = best_algo_name.get(grp_name, {}).get(dis_col)
        if algo is None:
            continue

        print(f"\n  [{grp_name}] [{dis_name}] — {algo} 5-fold CV 실행 중...")
        cv_result = run_cv(df_g, algo, dis_col)

        # hold-out 결과와 병합
        ho = all_results[grp_name][dis_col].get(algo, {})
        row = {
            '그룹':       grp_name,
            '질환':       dis_name,
            '최적알고리즘': algo,
            'HO_AUC':    ho.get('AUC', '-'),
            'HO_F1':     ho.get('F1', '-'),
            'HO_Precision': ho.get('Precision', '-'),
            'HO_Recall': ho.get('Recall', '-'),
            **{k: v for k, v in cv_result.items() if not k.startswith('_')},
        }
        cv_rows.append(row)
        print(f"    Hold-out AUC : {ho.get('AUC', '-')}")
        print(f"    CV AUC       : {cv_result['CV_AUC']}")
        print(f"    CV F1        : {cv_result['CV_F1']}")

# 최종 통합 표 저장 (논문 표 2)
cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv('성능비교_표2_최종.csv', index=False, encoding='utf-8-sig')
print("\n>>> 논문 표 2 저장 완료: 성능비교_표2_최종.csv")
print(cv_df[['그룹', '질환', '최적알고리즘',
             'HO_AUC', 'CV_AUC', 'CV_F1']].to_string(index=False))

# %%
# ## 5. 위험 확률값 추출 및 K-means 복합건강등급 산정
#
# 단계:
#   1) 각 그룹에서 최적 모델로 당뇨·고혈압 발병 확률 예측
#   2) 두 확률값을 2차원 피처로 K-means 군집화
#   3) 군집 중심의 위험도 순서로 1~K 등급 부여
#   4) 등급별 실제 유병률로 등급 타당성 검증

def assign_kmeans_grade(prob_dm: np.ndarray, prob_hp: np.ndarray,
                         n_clusters: int = 3, random_state: int = SEED):
    """
    당뇨·고혈압 예측 확률 2차원 벡터에 K-means 적용.
    군집 중심의 평균 확률 합산 순서로 등급 1(저위험) ~ n_clusters(고위험) 부여.
    반환: 등급 배열 (1-based)
    """
    X_km = np.column_stack([prob_dm, prob_hp])
    km   = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = km.fit_predict(X_km)

    # 군집 중심의 위험도 합산 기준으로 등급 순서 결정
    center_risk = km.cluster_centers_.sum(axis=1)
    rank_order  = np.argsort(center_risk)          # 낮은 위험 → 높은 위험 순
    grade_map   = {old: new + 1 for new, old in enumerate(rank_order)}
    grades      = np.vectorize(grade_map.get)(labels)
    return grades, km

# 최적 K 선정: 엘보우 곡선 (논문 부록용)
def plot_elbow(prob_dm: np.ndarray, prob_hp: np.ndarray,
               grp_name: str, k_range=range(2, 8)):
    X_km  = np.column_stack([prob_dm, prob_hp])
    inert = [KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_km).inertia_
             for k in k_range]
    plt.figure(figsize=(5, 3))
    plt.plot(list(k_range), inert, 'o-')
    plt.xlabel('군집 수 K')
    plt.ylabel('관성 (Inertia)')
    plt.title(f'[{grp_name}] 최적 K 엘보우 곡선')
    plt.tight_layout()
    plt.savefig(f'elbow_{grp_name}.png', dpi=150)
    plt.close()
    print(f"  엘보우 그래프 저장: elbow_{grp_name}.png")

grade_results = {}   # 그룹별 등급 결과 저장

for grp_name, cfg in AGEGROUP_CONFIG.items():
    print(f"\n{'='*20} [{grp_name}] 복합건강등급 산정 {'='*20}")

    df_g = df_final[
        (df_final['AgeGroup'] == cfg['age_group']) &
        (df_final['Sex']      == cfg['sex_code'])
    ].copy().reset_index(drop=True)

    X = df_g[X_FEATURES]

    mdl_dm = best_models[grp_name].get('Diabetes')
    mdl_hp = best_models[grp_name].get('Hypertension')

    if mdl_dm is None or mdl_hp is None:
        print("  ⚠️  최적 모델 없음. 건너뜀.")
        continue

    # 발병 확률 추출
    algo_dm = best_algo_name[grp_name]['Diabetes']
    algo_hp = best_algo_name[grp_name]['Hypertension']

    if algo_dm == 'TabNet':
        prob_dm = mdl_dm.predict_proba(X.values)[:, 1]
    else:
        prob_dm = mdl_dm.predict_proba(X)[:, 1]

    if algo_hp == 'TabNet':
        prob_hp = mdl_hp.predict_proba(X.values)[:, 1]
    else:
        prob_hp = mdl_hp.predict_proba(X)[:, 1]

    # 엘보우 곡선으로 K 시각화
    plot_elbow(prob_dm, prob_hp, grp_name)

    # K=3 으로 복합건강등급 산정
    # (엘보우 곡선 확인 후 K 조정 가능)
    N_CLUSTERS = 3
    grades, km_model = assign_kmeans_grade(prob_dm, prob_hp, n_clusters=N_CLUSTERS)

    df_g['prob_DM'] = prob_dm
    df_g['prob_HP'] = prob_hp
    df_g['복합건강등급'] = grades

    # 등급별 통계 (논문 표 3)
    summary = df_g.groupby('복합건강등급').agg(
        건수         = ('복합건강등급', 'count'),
        당뇨유병률   = ('Diabetes',     'mean'),
        고혈압유병률 = ('Hypertension', 'mean'),
        평균당뇨확률 = ('prob_DM',      'mean'),
        평균고혈압확률=('prob_HP',      'mean'),
    ).round(4)
    summary['구성비'] = (summary['건수'] / summary['건수'].sum()).round(4)
    print(summary.to_string())

    grade_results[grp_name] = df_g.copy()

    # 등급별 저장
    df_g.to_csv(f'등급결과_{grp_name}.csv', index=False, encoding='utf-8-sig')

# %%
# ## 6. SHAP 기반 변수 중요도 분석 (논문 그림 3–6)
#
# 각 그룹 × 질환별로:
#   1) Summary plot: 전역 변수 중요도 (bee-swarm)
#   2) 개별 관측치 waterfall plot (대표 1건)

def run_shap_analysis(model, X_val: pd.DataFrame,
                      algo: str, grp_name: str, dis_name: str,
                      n_sample: int = 500):
    """
    SHAP 분석 실행 및 시각화.
    n_sample: Summary plot용 배경 데이터 샘플 수 (속도·메모리 균형)
    """
    print(f"\n  SHAP 분석: [{grp_name}] [{dis_name}] (알고리즘: {algo})")

    X_s = X_val.sample(min(n_sample, len(X_val)), random_state=SEED)

    # TreeExplainer: RF, LGBM, XGB, TabNet(트리 불가→KernelExplainer 폴백)
    if algo in ('RF', 'LGBM', 'XGB'):
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_s)
        # RF·LGBM: shap_values가 리스트([음성, 양성]) 또는 3차원 배열로 반환됨
        # → 양성 클래스(인덱스 1)만 추출해 (n_samples, n_features) 2차원으로 정리
        if isinstance(shap_values, list):
            sv = shap_values[1]                     # 리스트 형태
        elif shap_values.ndim == 3:
            sv = shap_values[:, :, 1]               # (n, f, 2) → (n, f)
        else:
            sv = shap_values                        # XGB: 이미 2차원
    else:
        # MLP, TabNet → KernelExplainer (느리므로 샘플 축소)
        X_bg = shap.sample(X_s, min(100, len(X_s)))
        if algo == 'TabNet':
            predict_fn = lambda x: model.predict_proba(x)[:, 1]
            X_bg_val   = X_bg.values if hasattr(X_bg, 'values') else X_bg
            explainer   = shap.KernelExplainer(
                lambda x: model.predict_proba(x)[:, 1], X_bg_val
            )
            sv = explainer.shap_values(X_s.values[:50])
            X_s = pd.DataFrame(X_s.values[:50], columns=X_s.columns)
        else:
            explainer  = shap.KernelExplainer(
                lambda x: model.predict_proba(x)[:, 1], X_bg
            )
            sv = explainer.shap_values(X_s.iloc[:50])
            X_s = X_s.iloc[:50]

    # ── Summary plot (전역 변수 중요도) ──────────────────────
    plt.figure(figsize=(8, 6))
    shap.summary_plot(sv, X_s, plot_type='dot', show=False, max_display=15)
    plt.title('')   # 제목 제거 — 논문 캡션으로 대체
    plt.tight_layout()
    fname_sum = f'shap_summary_{grp_name}_{dis_name}.png'
    plt.savefig(fname_sum, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    저장: {fname_sum}")

    # ── 개별 관측치 Waterfall plot (대표 1건) ─────────────────
    # 예측 확률이 0.4–0.6 구간(경계선 사례)에서 1건 선택
    if algo in ('RF', 'LGBM', 'XGB'):
        probs = model.predict_proba(X_s)[:, 1]
    elif algo == 'TabNet':
        probs = model.predict_proba(X_s.values)[:, 1]
    else:
        probs = model.predict_proba(X_s)[:, 1]

    border_idx = np.where((probs >= 0.35) & (probs <= 0.65))[0]
    idx = border_idx[0] if len(border_idx) > 0 else 0

    # expected_value 스칼라 추출
    # RF: 배열([음성, 양성]) → 양성 클래스 인덱스 1
    # LGBM/XGB: 단일 float 또는 리스트
    ev = explainer.expected_value
    if isinstance(ev, (list, np.ndarray)):
        base_val = float(ev[1])
    else:
        base_val = float(ev)

    plt.figure(figsize=(8, 5))
    shap.waterfall_plot(
        shap.Explanation(
            values        = sv[idx],
            base_values   = base_val,
            data          = X_s.iloc[idx].values,
            feature_names = X_s.columns.tolist(),
        ),
        show=False, max_display=12,
    )
    plt.title('')   # 제목 제거 — 논문 캡션으로 대체
    plt.tight_layout()
    fname_ind = f'shap_individual_{grp_name}_{dis_name}.png'
    plt.savefig(fname_ind, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"    저장: {fname_ind}")

    return sv, X_s

# 4개 그룹 × 2개 질환 SHAP 분석 실행
shap_results = {}

for grp_name, cfg in AGEGROUP_CONFIG.items():
    shap_results[grp_name] = {}
    df_g = df_final[
        (df_final['AgeGroup'] == cfg['age_group']) &
        (df_final['Sex']      == cfg['sex_code'])
    ].copy()

    X = df_g[X_FEATURES]
    print(f"\n{'='*20} [{grp_name}] SHAP 분석 {'='*20}")

    for dis_col, dis_name in DISEASES.items():
        mdl  = best_models[grp_name].get(dis_col)
        algo = best_algo_name[grp_name].get(dis_col)
        if mdl is None:
            continue
        y = df_g[dis_col].astype(int)
        _, X_val, _, _ = train_test_split(
            X, y, test_size=0.3, random_state=SEED, stratify=y
        )
        sv, X_s = run_shap_analysis(mdl, X_val, algo, grp_name, dis_name)
        shap_results[grp_name][dis_col] = {'shap_values': sv, 'X_sample': X_s}

# %%
# ## 7. 연령군 간 SHAP 변수 중요도 비교 시각화 (논문 그림 7)
#
# 중장년 vs 고령, 남성 vs 여성 간 상위 변수 SHAP 절대값 평균 비교
# → 연령층별 위험 요인 차이를 직관적으로 보여주는 핵심 그래프

def compare_shap_across_groups(shap_results: dict, disease: str,
                                 top_n: int = 10):
    """
    4개 그룹의 SHAP 절대값 평균을 하나의 바 차트로 비교.
    disease: 'Diabetes' 또는 'Hypertension'
    """
    dis_label = {'Diabetes': '당뇨', 'Hypertension': '고혈압'}[disease]
    importance_dict = {}

    for grp_name in AGEGROUP_CONFIG:
        if grp_name not in shap_results:
            continue
        res = shap_results[grp_name].get(disease)
        if res is None:
            continue
        sv   = res['shap_values']
        cols = res['X_sample'].columns.tolist()
        mean_abs = np.abs(sv).mean(axis=0)
        importance_dict[grp_name] = pd.Series(mean_abs, index=cols)

    if not importance_dict:
        return

    # 전체 그룹 통합 상위 N개 변수 선정
    all_imp = pd.DataFrame(importance_dict).fillna(0)
    top_features = all_imp.mean(axis=1).nlargest(top_n).index.tolist()
    plot_df = all_imp.loc[top_features].T

    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df.plot(kind='bar', ax=ax, width=0.7)
    ax.set_title('')   # 제목 제거 — 논문 캡션으로 대체
    ax.set_xlabel('그룹')
    ax.set_ylabel('SHAP 절대값 평균')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    fname = f'shap_comparison_{disease}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  그룹 비교 그래프 저장: {fname}")

compare_shap_across_groups(shap_results, 'Diabetes')
compare_shap_across_groups(shap_results, 'Hypertension')

# %%
# ## 8. 복합건강등급 시각화 (논문 그림 8)
#
# 4개 그룹별 등급 분포 + 유병률 이중축 바 차트
# → 공모전 논문(연구1) 그림 14–18에 대응하는 업그레이드 버전

def plot_grade_distribution(df_grp: pd.DataFrame, grp_name: str):
    """등급별 구성비(막대) + 당뇨·고혈압 유병률(꺾은선) 이중축 그래프"""
    summary = df_grp.groupby('복합건강등급').agg(
        건수         = ('복합건강등급', 'count'),
        당뇨유병률   = ('Diabetes',     'mean'),
        고혈압유병률 = ('Hypertension', 'mean'),
    )
    summary['구성비'] = summary['건수'] / summary['건수'].sum()

    fig, ax1 = plt.subplots(figsize=(6, 4))
    x = summary.index
    ax1.bar(x, summary['구성비'], color='steelblue', alpha=0.7, label='구성비')
    ax1.set_xlabel('복합건강등급')
    ax1.set_ylabel('구성비', color='steelblue')
    ax1.set_ylim(0, 1)

    ax2 = ax1.twinx()
    ax2.plot(x, summary['당뇨유병률'],   'o-r', label='당뇨 유병률')
    ax2.plot(x, summary['고혈압유병률'], 's--g', label='고혈압 유병률')
    ax2.set_ylabel('유병률', color='black')
    ax2.set_ylim(0, 1)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
    plt.title('')   # 제목 제거 — 논문 캡션으로 대체
    plt.tight_layout()
    fname = f'grade_dist_{grp_name}.png'
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"  등급 분포 그래프 저장: {fname}")

for grp_name, df_g in grade_results.items():
    plot_grade_distribution(df_g, grp_name)

# %%
# ## 9. 모든 결과물 저장

# 최적 모델 저장
for grp_name in AGEGROUP_CONFIG:
    for dis_col in DISEASES:
        mdl = best_models.get(grp_name, {}).get(dis_col)
        if mdl is not None:
            joblib.dump(mdl, f'model_{grp_name}_{dis_col}.pkl')

# 등급 통합 CSV
all_grades = pd.concat(
    [df_g.assign(그룹=grp) for grp, df_g in grade_results.items()],
    ignore_index=True
)
all_grades.to_csv('복합건강등급_전체.csv', index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(">>> 전체 파이프라인 완료.")
print("  생성 파일 목록:")
print("  - 성능비교_표2.csv")
print("  - 복합건강등급_전체.csv")
print("  - 등급결과_{그룹}.csv (4개)")
print("  - shap_summary_{그룹}_{질환}.png (8개)")
print("  - shap_individual_{그룹}_{질환}.png (8개)")
print("  - shap_comparison_{질환}.png (2개)")
print("  - grade_dist_{그룹}.png (4개)")
print("  - elbow_{그룹}.png (4개)")
print("  - model_{그룹}_{질환}.pkl (최대 8개)")
print("="*50)

병합 후 전체 행수: 34,640
컬럼 수: 37

당뇨 분포:
 HE_DM_HbA1c
0.0    9046
1.0    2421
Name: count, dtype: int64

고혈압 분포:
 HE_HP
0.0    8267
1.0    3200
Name: count, dtype: int64

연령군 분포 (0=중장년 40–59세, 1=고령 60세+):
age_group
0    3897
1    3576
Name: count, dtype: int64
>>> 전처리 완료. 저장: knhanes_preprocessed.csv

df_final 형태: (7473, 35)

[연령군 × 성별 × 당뇨 분포]
Diabetes       0.0  1.0
AgeGroup Sex           
0.0      1.0   936  337
         2.0  2346  278
1.0      1.0   701  868
         2.0  1142  865

[연령군 × 성별 × 고혈압 분포]
Hypertension   0.0   1.0
AgeGroup Sex            
0.0      1.0   855   418
         2.0  2240   384
1.0      1.0   547  1022
         2.0   749  1258

==================== [중장년_남성] 표본 수: 1,273명 ====================

  ── [당뇨] 알고리즘 비교 ──
    RF       | AUC=0.7768 F1=0.7597 Prec=0.7559 Rec=0.7670
    LGBM     | AUC=0.7788 F1=0.7652 Prec=0.7942 Rec=0.7539
    XGB      | AUC=0.7840 F1=0.7412 Prec=0.7767 Rec=0.7277
    MLP      | AUC=0.5917 F1=0.6766 Prec=0.6655 Rec=0.7042

Early stopping occu

In [12]:
# ============================================================
# 논문 그림 3–8 생성 코드
# 그림 1·2는 PPT 별도 제작
#
# [그림 3] 알고리즘 AUC 성능 히트맵
# [그림 4] K-means 엘보우 곡선 (2×2)
# [그림 5] 복합건강등급 분포 (2×2)
# [그림 6] SHAP Summary — 당뇨 (2×2)
# [그림 7] SHAP Summary — 고혈압 (2×2)
# [그림 8] SHAP 집단 간 변수 중요도 비교
#
# ※ 메인 파이프라인 실행 후 연속 실행
#   필요 변수: all_results, df_final, best_models,
#              best_algo_name, grade_results, shap_results, X_FEATURES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import shap
from sklearn.cluster import KMeans

# ── 폰트·공통 설정 ──────────────────────────────────────────
plt.rcParams['font.family']            = 'Malgun Gothic'
matplotlib.rcParams['font.family']     = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus']     = False
matplotlib.rcParams['axes.unicode_minus'] = False

SEED      = 42
GRP_NAMES = ["중장년_남성", "중장년_여성", "고령_남성", "고령_여성"]
GRP_LABEL = ["중장년 남성", "중장년 여성", "고령 남성",  "고령 여성"]
COLORS    = ['#2196F3', '#E91E63', '#FF9800', '#4CAF50']
DPI       = 200


# ============================================================
# [그림 2] 집단별 당뇨·고혈압 유병률 비교 (2×2)
# ============================================================
def fig2_prevalence(save_path='그림2_집단별유병률.png'):
    cfg = {
        "중장년_남성": (0.0, 1.0),
        "중장년_여성": (0.0, 2.0),
        "고령_남성":   (1.0, 1.0),
        "고령_여성":   (1.0, 2.0),
    }

    fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharey=False)
    axes = axes.flatten()

    for ax, (grp, (age_g, sex_c)), color, lbl in \
            zip(axes, cfg.items(), COLORS, GRP_LABEL):

        df_g = df_final[
            (df_final['AgeGroup'] == age_g) &
            (df_final['Sex']      == sex_c)
        ]
        dm_r = df_g['Diabetes'].mean()    * 100
        hp_r = df_g['Hypertension'].mean()* 100
        n    = len(df_g)

        # alpha는 리스트 불가 → 막대별 개별 호출
        b1 = ax.bar(['당뇨'],     [dm_r], color=color, alpha=1.0,
                    edgecolor='white', linewidth=0.8, width=0.45)
        b2 = ax.bar(['고혈압'],   [hp_r], color=color, alpha=0.55,
                    edgecolor='white', linewidth=0.8, width=0.45)
        bars = list(b1) + list(b2)

        # 막대 위 수치 표기
        for bar, val in zip(bars, [dm_r, hp_r]):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.8,
                    f'{val:.1f}%',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

        ax.set_title(f'{lbl}  (n={n:,})', fontsize=11, pad=6)
        ax.set_ylim(0, 80)
        ax.set_ylabel('유병률 (%)', fontsize=10)
        ax.tick_params(axis='x', labelsize=11)
        ax.tick_params(axis='y', labelsize=9)
        ax.spines[['top', 'right']].set_visible(False)

        # 기준선
        ax.axhline(y=0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {save_path}")
def fig3_auc_heatmap(save_path='그림3_AUC히트맵.png'):
    algos = ['RF', 'LGBM', 'XGB', 'MLP', 'TabNet']
    rows, labels = [], []

    for grp, lbl in zip(GRP_NAMES, GRP_LABEL):
        for dis, dis_lbl in [('Diabetes', '당뇨'), ('Hypertension', '고혈압')]:
            aucs = []
            for algo in algos:
                m = all_results.get(grp, {}).get(dis, {}).get(algo)
                aucs.append(m['AUC'] if m else np.nan)
            rows.append(aucs)
            labels.append(f'{lbl} ({dis_lbl})')

    data = np.array(rows)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(data, cmap='RdYlGn', vmin=0.50, vmax=0.90, aspect='auto')
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('AUC', fontsize=10)

    ax.set_xticks(range(len(algos)))
    ax.set_xticklabels(algos, fontsize=11)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=10)

    # 셀 값 표기 — 최적 알고리즘 굵게
    for i in range(len(labels)):
        best_j = int(np.nanargmax(data[i]))
        for j in range(len(algos)):
            val = data[i, j]
            if np.isnan(val):
                continue
            txt_c  = 'white' if (val < 0.62 or val > 0.82) else 'black'
            weight = 'bold' if j == best_j else 'normal'
            ax.text(j, i, f'{val:.3f}',
                    ha='center', va='center',
                    fontsize=9, color=txt_c, fontweight=weight)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {save_path}")


# ============================================================
# [그림 4] K-means 엘보우 곡선 (2×2)
# ============================================================
def fig4_elbow(save_path='그림4_엘보우곡선.png'):
    cfg = {
        "중장년_남성": (0.0, 1.0),
        "중장년_여성": (0.0, 2.0),
        "고령_남성":   (1.0, 1.0),
        "고령_여성":   (1.0, 2.0),
    }

    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    axes = axes.flatten()

    for ax, (grp, (age_g, sex_c)), color, lbl in \
            zip(axes, cfg.items(), COLORS, GRP_LABEL):

        df_g = df_final[
            (df_final['AgeGroup'] == age_g) &
            (df_final['Sex']      == sex_c)
        ].copy()

        mdl_dm = best_models.get(grp, {}).get('Diabetes')
        mdl_hp = best_models.get(grp, {}).get('Hypertension')
        if mdl_dm is None or mdl_hp is None:
            ax.set_title(lbl); ax.axis('off'); continue

        a_dm   = best_algo_name.get(grp, {}).get('Diabetes', '')
        a_hp   = best_algo_name.get(grp, {}).get('Hypertension', '')
        X      = df_g[X_FEATURES]
        prob_dm = mdl_dm.predict_proba(
            X.values if a_dm == 'TabNet' else X)[:, 1]
        prob_hp = mdl_hp.predict_proba(
            X.values if a_hp == 'TabNet' else X)[:, 1]

        X_km   = np.column_stack([prob_dm, prob_hp])
        k_rng  = range(2, 8)
        inerts = [KMeans(n_clusters=k, random_state=SEED, n_init=10)
                  .fit(X_km).inertia_ for k in k_rng]

        ax.plot(list(k_rng), inerts, 'o-', color=color, lw=2, ms=6)
        ax.axvline(x=3, color='red', linestyle='--', lw=1.2, alpha=0.8)
        ax.text(3.1, max(inerts) * 0.95, 'K=3',
                color='red', fontsize=9)
        ax.set_xlabel('군집 수 K', fontsize=10)
        ax.set_ylabel('관성 (Inertia)', fontsize=10)
        ax.set_title(lbl, fontsize=11)
        ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {save_path}")


# ============================================================
# [그림 5] 복합건강등급 분포 (2×2)
# ============================================================
def fig5_grade_dist(save_path='그림5_복합건강등급분포.png'):
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.flatten()

    for ax, grp, color, lbl in zip(axes, GRP_NAMES, COLORS, GRP_LABEL):
        df_g = grade_results.get(grp)
        if df_g is None:
            ax.set_title(lbl); ax.axis('off'); continue

        summary = df_g.groupby('복합건강등급').agg(
            건수         = ('복합건강등급', 'count'),
            당뇨유병률   = ('Diabetes',     'mean'),
            고혈압유병률 = ('Hypertension', 'mean'),
        )
        summary['구성비'] = summary['건수'] / summary['건수'].sum() * 100
        grades = summary.index.tolist()

        ax2 = ax.twinx()
        ax.bar(grades, summary['구성비'],
               color=color, alpha=0.4, label='구성비', zorder=2)
        ax2.plot(grades, summary['당뇨유병률'] * 100,
                 'o-r', lw=2, ms=7, label='당뇨 유병률', zorder=3)
        ax2.plot(grades, summary['고혈압유병률'] * 100,
                 's--b', lw=2, ms=7, label='고혈압 유병률', zorder=3)

        # 수치 표기
        for g, dm, hp in zip(grades,
                              summary['당뇨유병률'] * 100,
                              summary['고혈압유병률'] * 100):
            ax2.text(g, dm + 2, f'{dm:.1f}%',
                     ha='center', fontsize=8, color='red')
            ax2.text(g, max(hp - 6, 2), f'{hp:.1f}%',
                     ha='center', fontsize=8, color='blue')

        ax.set_xlabel('복합건강등급', fontsize=10)
        ax.set_ylabel('구성비 (%)', fontsize=10, color=color)
        ax2.set_ylabel('유병률 (%)', fontsize=10)
        ax2.set_ylim(0, 105)
        ax.set_ylim(0, 70)
        ax.set_title(lbl, fontsize=11)
        ax.set_xticks(grades)
        ax.set_xticklabels(
            ['1등급\n(저위험)', '2등급\n(중위험)', '3등급\n(고위험)'],
            fontsize=9)
        ax.spines[['top']].set_visible(False)

        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper left')

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {save_path}")


# ============================================================
# [그림 6·7] SHAP Summary Plot — 당뇨 / 고혈압 (각 2×2)
# ============================================================
def fig6_7_shap_summary(disease, dis_label, fig_num,
                         save_path_tpl='그림{n}_SHAP_Summary_{d}.png'):
    """
    SHAP summary_plot의 내장 컬러바를 완전히 차단하고
    수동으로 외부에 단일 공통 컬러바를 배치.
    핵심: summary_plot을 호출한 직후 ax 내부에 생긴
    컬러바 axes를 찾아서 강제 제거.
    """
    # 여유 있는 figsize — 우측 컬러바 열 별도 확보
    fig = plt.figure(figsize=(15, 10))

    # 2×2 서브플롯 영역 (전체 너비의 88%만 사용)
    gs = fig.add_gridspec(2, 2,
                          left=0.08, right=0.86,
                          top=0.93, bottom=0.07,
                          wspace=0.55, hspace=0.50)

    for idx, (grp, lbl) in enumerate(zip(GRP_NAMES, GRP_LABEL)):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])

        res = shap_results.get(grp, {}).get(disease)
        if res is None:
            ax.set_title(lbl, fontsize=11)
            ax.axis('off')
            continue

        sv  = res['shap_values']
        X_s = res['X_sample']

        # ── summary_plot 호출 (color_bar=False 로 내장 컬러바 끄기) ──
        plt.sca(ax)
        shap.summary_plot(sv, X_s,
                          plot_type='dot',
                          show=False,
                          max_display=12,
                          color_bar=False)   # ← 내장 컬러바 완전 비활성

        ax.set_title(lbl, fontsize=11, pad=6)
        ax.tick_params(axis='both', labelsize=8)

        # ── summary_plot이 몰래 추가한 컬러바 axes 강제 제거 ──
        # figure 내 axes 목록에서 현재 서브플롯 ax가 아닌
        # 폭이 매우 좁은 axes(컬러바)를 탐지해 삭제
        for extra_ax in fig.axes:
            if extra_ax is ax:
                continue
            pos = extra_ax.get_position()
            # 컬러바 axes는 너비가 0.05 미만이고 높이가 작음
            if pos.width < 0.05:
                extra_ax.remove()

    # ── 수동 공통 컬러바 (figure 우측 고정 위치) ──
    cbar_ax = fig.add_axes([0.88, 0.20, 0.018, 0.55])
    sm = plt.cm.ScalarMappable(
        cmap=shap.plots.colors.red_blue,
        norm=plt.Normalize(vmin=0, vmax=1)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('특성값', fontsize=9, labelpad=8, rotation=270, va='bottom')
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['낮음', '중간', '높음'], fontsize=8)

    fname = save_path_tpl.format(n=fig_num, d=dis_label)
    plt.savefig(fname, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {fname}")


# ============================================================
# [그림 8] SHAP 집단 간 변수 중요도 비교 (당뇨·고혈압 나란히)
# ============================================================
def fig8_shap_comparison(top_n=10,
                          save_path='그림8_SHAP집단비교.png'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, disease, dis_lbl in zip(
        axes,
        ['Diabetes', 'Hypertension'],
        ['당뇨', '고혈압']
    ):
        imp_dict = {}
        for grp, lbl in zip(GRP_NAMES, GRP_LABEL):
            res = shap_results.get(grp, {}).get(disease)
            if res is None:
                continue
            sv   = res['shap_values']
            cols = res['X_sample'].columns.tolist()
            imp_dict[lbl] = pd.Series(
                np.abs(sv).mean(axis=0), index=cols)

        if not imp_dict:
            continue

        all_imp   = pd.DataFrame(imp_dict).fillna(0)
        top_feats = all_imp.mean(axis=1).nlargest(top_n).index.tolist()
        plot_df   = all_imp.loc[top_feats]

        x     = np.arange(len(top_feats))
        width = 0.18
        for i, (col, color) in enumerate(
                zip(plot_df.columns, COLORS)):
            ax.barh(x + i * width, plot_df[col].values,
                    height=width, label=col,
                    color=color, alpha=0.85)

        ax.set_yticks(x + width * 1.5)
        ax.set_yticklabels(top_feats, fontsize=9)
        ax.set_xlabel('SHAP 절대값 평균', fontsize=10)
        ax.set_title(f'{dis_lbl} 예측 — 집단별 변수 중요도', fontsize=11)
        ax.legend(fontsize=9, loc='lower right')
        ax.spines[['top', 'right']].set_visible(False)
        ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(save_path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"저장: {save_path}")


# ============================================================
# 실행
# ============================================================
print("\n" + "="*50)
print("논문 그림 2–8 생성 시작")
print("="*50)

fig2_prevalence()
fig3_auc_heatmap()
fig4_elbow()
fig5_grade_dist()
fig6_7_shap_summary('Diabetes',     '당뇨',  6)
fig6_7_shap_summary('Hypertension', '고혈압', 7)
fig8_shap_comparison()

print("\n완료. 생성 파일:")
for i, f in enumerate([
    "그림2_집단별유병률.png",
    "그림3_AUC히트맵.png",
    "그림4_엘보우곡선.png",
    "그림5_복합건강등급분포.png",
    "그림6_SHAP_Summary_당뇨.png",
    "그림7_SHAP_Summary_고혈압.png",
    "그림8_SHAP집단비교.png",
], 2):
    print(f"  [그림 {i}] {f}")
print("="*50)


논문 그림 2–8 생성 시작
저장: 그림2_집단별유병률.png
저장: 그림3_AUC히트맵.png
저장: 그림4_엘보우곡선.png
저장: 그림5_복합건강등급분포.png
저장: 그림6_SHAP_Summary_당뇨.png
저장: 그림7_SHAP_Summary_고혈압.png
저장: 그림8_SHAP집단비교.png

완료. 생성 파일:
  [그림 2] 그림2_집단별유병률.png
  [그림 3] 그림3_AUC히트맵.png
  [그림 4] 그림4_엘보우곡선.png
  [그림 5] 그림5_복합건강등급분포.png
  [그림 6] 그림6_SHAP_Summary_당뇨.png
  [그림 7] 그림7_SHAP_Summary_고혈압.png
  [그림 8] 그림8_SHAP집단비교.png
